<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

</br>

In [ ]:
# TODO 0: 실습을 위해 아래 패키지를 import 해주세요.
import copy
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.models import resnet18, ResNet18_Weights
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 디바이스: {device}")

</br>

# 학습 내용
>이번 장에서는 <strong>Fine-tuning과 데이터 증강(Fine-tuning & Data Augmentation)</strong>에 대해 학습합니다.</br></br>
>모델 전체를 미세 조정하고, 데이터 증강으로 학습 데이터를 학습해봅시다.

</br>

# Fine-tuning (미세 조정)
> 사전학습 모델의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">모든 가중치를 동결 해제(Unfreeze)</mark>하고 새 태스크에 맞게 재학습합니다.

<mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Linear Probing</mark>은 사전학습된 백본을 완전히 동 결하고 마지막 분류 헤드만 학습하는 반면, **Fine-tuning**은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">백본 전체의 가중치를 함께 업데이트</mark>하여 새로운 태스크에 맞게 모델 전체를 적응시킵니다. 실제 학습 데이터는 대부분 부족하므로, 과적합을 방지하고 일반화 성능을 높이려면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;"> 회전 / 좌우 반전 / 색상 변환</mark> 등의 변환으로 동일한 이미지를 다양하게 변형하여 데이터 다양성을 인위적으로 확보하는 **데이터 증강**이 함께 필요합니다.</br>이 장에서는 전이학습의 개념, Linear Probing과의 비교, Fine-tuning 시 사전학습 가중치를 보존하기  위한 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">학습률(Learning Rate)</mark> 설정(예: 1e-4), 그리고 학습 데이터에만 특화되어 새 데이터에서 성능이 저하되는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">과적합(Overfitting)</mark> 현상을 다룹니다.

In [ ]:
# TODO 1: 모델의 모든 파라미터에 대해 동결을 해제하고, 학습 가능한 파라미터 수를 출력해봅시다.

for param in model.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"학습 파라미터: {trainable:,} / {total:,} ({trainable/total:.0%})")

</br>

## Linear Probing vs Fine-tuning

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">비교 항목</th>
      <th>Linear Probing</th>
      <th>Fine-tuning</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">학습 범위</td><td>fc 레이어만</td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">전체 모델</mark></td></tr>
    <tr><td style="text-align:center">학습 속도</td><td>빠름</td><td>느림</td></tr>
    <tr><td style="text-align:center">필요 데이터량</td><td>적음</td><td>많음</td></tr>
    <tr><td style="text-align:center">과적합 위험</td><td>낮음</td><td>높음</td></tr>
    <tr><td style="text-align:center">최종 성능</td><td>보통</td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">일반적으로 높음</mark></td></tr>
  </tbody>
</table>

💡학습률 차이
> Fine-tuning 시 사전학습 레이어는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">작은 학습률</mark>(1e-4~1e-5)을 사용합니다.</br>
> 새로 추가한 fc 레이어는 상대적으로 큰 학습률(1e-3)을 사용합니다.

</br>

# 데이터 증강 (Data Augmentation)
> 학습 이미지에 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">랜덤 변형을 적용</mark>하여 데이터 다양성을 높이고 과적합을 방지합니다.

In [ ]:
# TODO 2: 학습용/검증용 전처리 파이프라인을 구성해봅시다.

# TODO 2-1: 학습용 전처리에 RandomResizedCrop(224)을 추가해봅시다.
# TODO 2-2: RandomHorizontalFlip(p=0.5)을 추가해봅시다.
# TODO 2-3: RandomRotation(15)을 추가해봅시다.
# TODO 2-4: ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)를 추가해봅시다.
# TODO 2-5: ToTensor()와 ImageNet Normalize를 추가해봅시다.

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# TODO 2-6: 검증용 전처리에는 Resize(256), CenterCrop(224), ToTensor, Normalize만 적용해봅시다.
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

print(f"학습 전처리: {train_transform}")
print(f"검증 전처리: {val_transform}")

</br>

## 주요 증강 기법

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">기법</th>
      <th>설명</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>RandomCrop</code></td><td>랜덤 위치에서 자르기</td></tr>
    <tr><td style="text-align:center"><code>RandomHorizontalFlip</code></td><td>좌우 반전 (p=0.5)</td></tr>
    <tr><td style="text-align:center"><code>RandomRotation</code></td><td>랜덤 회전</td></tr>
    <tr><td style="text-align:center"><code>ColorJitter</code></td><td>밝기, 대비, 채도 변형</td></tr>
    <tr><td style="text-align:center"><code>RandomErasing</code></td><td>랜덤 영역 마스킹</td></tr>
  </tbody>
</table>

💡검증 데이터에 증강 금지
> 검증과 테스트 데이터에는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">증강을 적용하지 않습니다</mark>.</br>
> Resize + CenterCrop + Normalize만 적용하여 일관된 평가를 보장합니다.

💡어떤 증강을 선택할까?
> 도메인에 맞는 변형만 적용합니다.</br>
> 예: 의료 이미지에서 좌우 반전은 의미가 있지만, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">상하 반전은 부적절</mark>할 수 있습니다.

</br>

# 데이터 준비
> CIFAR-10 데이터셋을 로드하고, 앞서 정의한 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">데이터 증강 파이프라인</mark>을 적용하여 학습/테스트 DataLoader를 구성합니다.

In [ ]:
# TODO 3: train_dataset에 CIFAR-10 학습 데이터를 로드해봅시다. (train_transform 적용)
train_dataset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=train_transform)

In [ ]:
# TODO 4: test_dataset에 CIFAR-10 테스트 데이터를 로드해봅시다. (val_transform 적용)
test_dataset = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=val_transform)

In [ ]:
# TODO 5: train_loader를 생성해봅시다. (batch_size=256, shuffle=True)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=2)

In [ ]:
# TODO 6: test_loader를 생성해봅시다. (batch_size=256, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=2)

print(f"학습: {len(train_dataset)}개, 테스트: {len(test_dataset)}개")

</br>

# Full Fine-tuning 학습
> 사전학습된 ResNet18의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">모든 레이어를 동결 해제(Unfreeze)</mark>하고, CIFAR-10 데이터셋에 맞게 전체 모델을 재학습합니다.</br></br>
> 모든 가중치가 업데이트되므로 표현력이 높지만, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">학습 시간이 길고 과적합 위험</mark>이 있습니다.

In [ ]:
# TODO 7: ResNet18 사전학습 모델을 로드해봅시다.
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

In [ ]:
# TODO 8: fc 레이어를 10 클래스에 맞게 교체해봅시다.
model.fc = nn.Linear(model.fc.in_features, 10)

In [ ]:
# TODO 9: 모든 파라미터의 동결을 해제해봅시다.
for param in model.parameters():
    param.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"학습 파라미터: {trainable_params:,}")

model = model.to(device)

In [ ]:
# TODO 10: criterion에 CrossEntropyLoss를 저장해봅시다.
criterion = nn.CrossEntropyLoss()

In [ ]:
# TODO 11: optimizer에 SGD를 저장해봅시다. (lr=0.0005, momentum=0.9)
optimizer = optim.SGD(model.parameters(), lr=0.0005, momentum=0.9)

In [ ]:
# TODO 12: scheduler에 StepLR을 저장해봅시다. (step_size=5, gamma=0.1)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

In [ ]:
# TODO 13: Full Fine-tuning 학습 루프 (5 epochs) + 평가

num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {running_loss/len(train_loader):.4f}, Acc: {100.*correct/total:.2f}%")

# 평가
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
full_ft_accuracy = 100. * correct / total
print(f"\nFull Fine-tuning 테스트 정확도: {full_ft_accuracy:.2f}%")

</br>

# Partial Fine-tuning (layer4 + fc)
> 사전학습된 ResNet18에서 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">layer1~layer3은 동결</mark>하고, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">layer4와 fc만 학습</mark>합니다.</br></br>
> 낮은 레이어(layer1~3)는 일반적인 시각 특성(edge, texture)을 이미 잘 학습한 상태이므로 보존하고, 높은 레이어(layer4)만 새 태스크에 맞게 재조정합니다.

In [ ]:
# TODO 14: 새로운 ResNet18 사전학습 모델을 로드해봅시다.
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

In [ ]:
# TODO 15: fc 레이어를 10 클래스에 맞게 교체해봅시다.
model.fc = nn.Linear(model.fc.in_features, 10)

In [ ]:
# TODO 16: 모든 파라미터를 동결해봅시다.
for param in model.parameters():
    param.requires_grad = False

In [ ]:
# TODO 17: model.layer4의 파라미터를 동결 해제해봅시다.
for param in model.layer4.parameters():
    param.requires_grad = True

In [ ]:
# TODO 18: model.fc의 파라미터를 동결 해제해봅시다.
for param in model.fc.parameters():
    param.requires_grad = True

In [ ]:
# TODO 19: 학습 가능한 파라미터 수를 출력해봅시다.
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"학습 파라미터: {trainable_params:,} / {total_params:,} ({trainable_params/total_params:.1%})")

model = model.to(device)

💡Partial Fine-tuning
> CNN의 낮은 레이어(layer1~3)는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">일반적인 특성(edge, texture)</mark>을 학습하여 도메인 간 공유가 가능합니다.</br>
> 높은 레이어(layer4)는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">태스크에 특화된 특성</mark>을 학습하므로, 새 태스크에 맞게 재조정이 필요합니다.

</br>

# Optimizer / Scheduler 비교
> Partial Fine-tuning 모델에 서로 다른 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Optimizer + Scheduler 조합</mark>을 적용하여 학습 성능을 비교합니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">조합</th>
      <th style="text-align:center">Optimizer</th>
      <th style="text-align:center">특징</th>
      <th style="text-align:center">Scheduler</th>
      <th style="text-align:center">특징</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align:center"><strong>Baseline</strong></td>
      <td style="text-align:center">SGD + Momentum</td>
      <td style="text-align:center">안정적 수렴, 일반화 우수</td>
      <td style="text-align:center">StepLR</td>
      <td style="text-align:center">고정 주기로 학습률 감소</td>
    </tr>
    <tr>
      <td style="text-align:center"><strong>Experiment</strong></td>
      <td style="text-align:center">Adam</td>
      <td style="text-align:center">적응적 학습률, 빠른 수렴</td>
      <td style="text-align:center">ReduceLROnPlateau</td>
      <td style="text-align:center">손실 기반 적응적 감소</td>
    </tr>
  </tbody>
</table>

In [ ]:
# TODO 20: initial_state에 모델 초기 상태를 저장해봅시다. (copy.deepcopy)
initial_state = copy.deepcopy(model.state_dict())

In [ ]:
# TODO 21: criterion에 CrossEntropyLoss를 저장해봅시다.
criterion = nn.CrossEntropyLoss()

In [ ]:
# TODO 22: optimizer에 SGD를 저장해봅시다. (lr=0.001, momentum=0.9)
optimizer = optim.SGD(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001, momentum=0.9)

In [ ]:
# TODO 23: scheduler에 StepLR을 저장해봅시다. (step_size=5, gamma=0.1)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

In [ ]:
# TODO 24: Baseline 학습 루프 (10 epochs) + 평가

num_epochs = 10
baseline_train_losses = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    scheduler.step()
    epoch_loss = running_loss / len(train_loader)
    baseline_train_losses.append(epoch_loss)
    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {epoch_loss:.4f}")

model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        _, predicted = model(images).max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
baseline_accuracy = 100. * correct / total
print(f"Baseline (SGD+StepLR) 정확도: {baseline_accuracy:.2f}%")

In [ ]:
# TODO 25: 초기 상태로 모델을 복원해봅시다.
model.load_state_dict(initial_state)

In [ ]:
# TODO 26: optimizer에 Adam을 저장해봅시다. (lr=0.001)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

In [ ]:
# TODO 27: scheduler에 ReduceLROnPlateau를 저장해봅시다. (mode="min", factor=0.1, patience=2)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.1, patience=2)

In [ ]:
# TODO 28: Experiment 학습 루프 (10 epochs) + 평가

num_epochs = 10
experiment_train_losses = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    epoch_loss = running_loss / len(train_loader)
    experiment_train_losses.append(epoch_loss)

    # TODO 28-1: ReduceLROnPlateau는 epoch_loss를 전달하여 스케줄러를 업데이트해봅시다.
    scheduler.step(epoch_loss)

    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {epoch_loss:.4f}")

model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        _, predicted = model(images).max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
experiment_accuracy = 100. * correct / total
print(f"Experiment (Adam+ReduceLR) 정확도: {experiment_accuracy:.2f}%")

</br>

# 학습 곡선 비교
> 두 가지 Optimizer/Scheduler 조합의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">학습 곡선(Train Loss)</mark>과 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">테스트 정확도</mark>를 시각적으로 비교합니다.

In [ ]:
# TODO 29: 학습 곡선을 비교 시각화해봅시다.

plt.figure(figsize=(10, 5))
plt.plot(baseline_train_losses, label=f"SGD+StepLR (Acc: {baseline_accuracy:.1f}%)", marker="o")
plt.plot(experiment_train_losses, label=f"Adam+ReduceLR (Acc: {experiment_accuracy:.1f}%)", marker="s")
plt.xlabel("Epoch")
plt.ylabel("Train Loss")
plt.title("Optimizer/Scheduler 비교")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

💡실험 결과 해석
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">수렴 속도(Convergence Speed)</mark>: 초반 에포크에서 loss가 빠르게 감소하는 조합이 수렴이 빠릅니다. Adam은 적응적 학습률 덕분에 초반 수렴이 빠른 경향이 있습니다.</br>
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">최종 손실(Final Loss)</mark>: 학습 종료 시점의 loss가 낮을수록 모델이 학습 데이터를 잘 피팅했음을 의미합니다. 단, 테스트 정확도와 함께 비교해야 과적합 여부를 판단할 수 있습니다.</br>
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">안정성(Stability)</mark>: loss 곡선의 진동이 클수록 학습이 불안정함을 의미합니다. SGD+Momentum은 일반적으로 더 안정적인 학습 곡선을 보입니다.